In [1]:
"""Copied from https://github.com/microsoft/qdk-chemistry/blob/main/examples/sossa_qre.ipynb."""

import math

import numpy as np
from qdk.qre.application import QSharpApplication
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef,
    Configuration,
    DFTHCHamiltonianContainer,
    Hamiltonian,
    MajoranaMapping,
    ModelOrbitals,
    StateVectorContainer,
    Wavefunction,
)


def get_sossa_app(small_circuit: bool = False) -> QSharpApplication:
    FE2S2_PARAMETERS = {
        "electrons": 30,
        "N": 20,
        "R": 14,
        "B": 15,
        "C": 5,
        "lambda_eff": 6.4690,
        "b_coeff": 11,
        "b_rot": 15,
    }
    TARGET_PRECISION = 1e-3
    seed = 42
    fe2s2_num_queries = math.ceil(
        math.pi * FE2S2_PARAMETERS["lambda_eff"] / (2 * TARGET_PRECISION)
    )

    if small_circuit:
        FE2S2_PARAMETERS.update(electrons=2, N=2, R=2, B=2, C=2, b_coeff=3, b_rot=3)
        fe2s2_num_queries = 3

    n, r, b, c = (FE2S2_PARAMETERS[key] for key in ("N", "R", "B", "C"))
    rng = np.random.default_rng(seed)
    one_body = rng.standard_normal((n, n))
    basis_vectors = rng.standard_normal((r, b, n))
    basis_vectors /= np.linalg.norm(basis_vectors, axis=-1, keepdims=True)
    synthetic_fe2_s2_hamiltonian = Hamiltonian(
        DFTHCHamiltonianContainer(
            one_body_integrals=(one_body + one_body.T) / 2,
            u_matrices=basis_vectors.ravel(),
            w_matrices=0.1 * rng.standard_normal((r, b, c)).ravel(),
            wb_matrix=0.1 * rng.standard_normal((r, c)),
            orbitals=ModelOrbitals(n),
            core_energy=0.0,
            inactive_fock_matrix=np.zeros((n, n)),
        )
    )
    fe2s2_operator = create("qubit_mapper", "sum_of_squares").run(
        synthetic_fe2_s2_hamiltonian, MajoranaMapping.jordan_wigner(2 * n)
    )

    container = synthetic_fe2_s2_hamiltonian.get_container()
    configuration = Configuration.canonical_hf_configuration(
        (FE2S2_PARAMETERS["electrons"] + 1) // 2,
        FE2S2_PARAMETERS["electrons"] // 2,
        container.get_num_orbitals(),
    )
    reference = Wavefunction(
        StateVectorContainer(configuration, container.get_orbitals())
    )
    fe2s2_state_preparation = create("state_prep", "sparse_isometry").run(reference)

    sossa_unitary_builder = AlgorithmRef("hamiltonian_unitary_builder", "sossa")
    fe2s2_circuit = create(
        "qpe_circuit_builder",
        "qdk_unary",
        num_queries=fe2s2_num_queries,
        circuit_mapper=AlgorithmRef(
            "circuit_mapper",
            "sossa",
            outer_prepare_algorithm=AlgorithmRef("state_prep", "alias_sampling"),
            inner_prepare_algorithm="controlled_alias_sampling",
            select_algorithm="qrom_phase_gradient",
            rotation_bit_precision=FE2S2_PARAMETERS["b_rot"],
            coefficient_bit_precision=FE2S2_PARAMETERS["b_coeff"],
        ),
        unitary_builder=sossa_unitary_builder,
    ).run(
        state_preparation=fe2s2_state_preparation,
        qubit_hamiltonian=fe2s2_operator,
    )[
        0
    ]
    return fe2s2_circuit.get_qre_application()

In [2]:
from qdk import Context, qsharp
from qdk.qre import DemoModel1, SequentialCountingProfiler, TemporalProfiler, TemporalProfilerConfig, estimate
from qdk.qre.models import GateBased, RoundBasedFactory, SurfaceCode, TwoDimensionalYokedSurfaceCode
from qdk.qre.application import QSharpApplication, QIRApplication
from qdk.qre.profiler import QREProfiler

def make_profiler(record_trace=False, aggregation_window_sec=1.0,t_factories_count=10):
    model = DemoModel1(
        clifford_gate_time_nanos=5000,
        measurment_time_nanos=1000,    
        t_factories_count=t_factories_count,
        ecc_distance=9,
    )
    config = TemporalProfilerConfig(
        seed=42,
        record_trace=record_trace,
        aggregation_window_nanos = int(aggregation_window_sec*1e9),
    )
    profiler = TemporalProfiler(model, config)
    profiler.add_adapter("transpiler")
    profiler.add_adapter("result_adapter")
    return QREProfiler(profiler)

In [3]:
app = get_sossa_app(small_circuit=True)
profiler = make_profiler()
profiler.profile(app)

[2026-11-10 13:45:06.781425] [info] [qdk_chemistry.algorithms.state_preparation.sparse_isometry] After filtering, only 1 determinant remains, using single reference state preparation


RuntimeError: Rz angle 0.46364760900080604 is not within 0.0000000001 radians of +/- 2^k * pi

In [ ]:
qsharp.compile(app.entry_expr, app.args)